In [1]:
#from data_quality_test_management import *
from recipe_management import *
from dotenv import load_dotenv
import os
# Charger les variables du fichier .env
load_dotenv()
from rapidfuzz import fuzz, process
import re




In [3]:
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY_2')
MODEL_NAME = os.getenv('MODEL_NAME_2')

In [ ]:
# jumelage des cohorts et creation des donnees vaec non geste et avec transformation de non geste en periode
df,df1 = process_files_smart(1,108,batch_size = 15)

CONFIGURATION DU TRAITEMENT
CPUs disponibles: 12
Processus utilisés: 6
Taille des lots: 15 fichiers
Cohortes à traiter: 106
Intervalle: [1 - 108]
Liste des cohortes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...


LOT 1/8
Cohortes traitées dans ce lot: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Nombre de fichiers: 15
------------------------------------------------------------

✓ Lot 1 terminé!
  Temps écoulé: 30.26s (2.02s par fichier)
  Lignes ajoutées: 579,118 (final) + 144,783 (temporal)

⏱  Temps estimé restant: 3.1 minutes
  Progression: 14.2%
  Pause de 3 secondes...

LOT 2/8
Cohortes traitées dans ce lot: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Nombre de fichiers: 15
------------------------------------------------------------

✓ Lot 2 terminé!
  Temps écoulé: 29.28s (1.95s par fichier)
  Lignes ajoutées: 577,864 (final) + 144,475 (temporal)

⏱  Temps estimé restant: 2.5 minutes
  Progression: 28.3%
  Pause de 3 secondes...

LOT 3/8
Cohortes traitées dans ce 

: 

In [5]:
def data_preparation_3stages_1(file_path):
    
    with open(file_path, 'r') as f:
        save_data = json.load(f)
        results = save_data.get('results', save_data)
    
    # Charger les données de base et créer un dictionnaire
    recipes = pd.read_csv("recipes.csv", usecols=['id', 'title'])
    recipe_titles = dict(zip(recipes['id'], recipes['title']))
    
    # Pré-allouer les listes avec une estimation de taille
    estimated_size = len(results['variantes_principales']) * 5
    all_data = []
    temporal_data = []
    
    # Définir les configurations de variantes pour éviter la répétition de code
    variante_configs = [
        ('variantes_principales', None, 'principal', 'variante_principale'),
        ('variantes_secondaires', 'ingredient_variant', 'secondaire', 'variante_ingredients'),
        ('variantes_secondaires', 'permutation_1', 'secondaire', 'variante_permutation'),
        ('variantes_secondaires', 'permutation_2', 'secondaire', 'variante_permutation'),
    ]
    
    # Traitement de chaque recette
    for recipe_id in results['variantes_principales'].keys():
        recipe_title = recipe_titles.get(recipe_id, f"Recipe_{recipe_id}")
        
        # Traiter les variantes principales et secondaires
        for result_key, sub_key, type_val, type_2_val in variante_configs:
            if sub_key is None:
                variante = results[result_key].get(recipe_id, [])
            else:
                variante = results[result_key].get(recipe_id, {}).get(sub_key, [])
            
            if variante and variante != ["NA"]:
                all_data.append({
                    'id': recipe_id,
                    'title': recipe_title,
                    'actions': variante,
                    'type': type_val,
                    'type_2': type_2_val
                })
        
        # Variante ternaire (temporelle)
        variante_ternaire = results['variantes_ternaires'].get(recipe_id, [])
        if variante_ternaire and variante_ternaire != ["NA"]:
            temporal_data.append({
                'id': recipe_id,
                'title': recipe_title,
                'actions': variante_ternaire,
                'type': 'principal'
            })
    
    # Créer les DataFrames une seule fois
    final_df = pd.DataFrame(all_data)
    temporal_df = pd.DataFrame(temporal_data)

    #final_df.to_csv('dataset_finaux/dataset_final.csv', index=False)
    #temporal_df.to_csv('dataset_finaux/dataset_temporel_final.csv', index=False)
    
    return final_df, temporal_df
suffixes = ['X','50_to_80_','80_to_100_','101_to_108']
for suffix in suffixes:
    if suffix == 'X':
        file_path = f"recipes_variants_3stages/sauvegarde_final_3stages_cohort_{suffix}.json"
    else:
        file_path = f"recipes_variants_3stages/retry_na_results_cohorts_{suffix}.json"
    dat,dat1 = data_preparation_3stages_1(file_path)
    df = pd.concat([df,dat])
    df1 = pd.concat([df1,dat1])
df.to_csv('data/dataset_graph_recette.csv')
df1.to_csv('data/dataset_graph_recette_avec_temps.csv')   

In [13]:
#actions_to_remove_final.extend([x for x in actions_to_remove if x not in actions_to_remove_final])
normalization_dict_final.update({k: v for k, v in normalization_dict.items() if k not in normalization_dict_final})

In [14]:
print(len(actions_to_remove_final))
print(len(normalization_dict_final))

2649
2300


In [10]:

with open('actions_to_remove_final.txt', 'r', encoding='utf-8') as f:
   actions_to_remove_final = [line.strip() for line in f if line.strip()]

    # Importer normalization_dict depuis le fichier JSON
with open('normalization_dict_final.json', 'r', encoding='utf-8') as f:
    normalization_dict_final = json.load(f)

In [15]:
# Sauvegarder normalization_dict dans un fichier JSON
with open('normalization_dict_final.json', 'w', encoding='utf-8') as f:
    json.dump(normalization_dict_final, f, ensure_ascii=False, indent=2)

In [6]:
# Vérifier verbes uniques après nettoyage
def extraire_verbes_uniques_from_col(df, col="actions"):
    s = set()
    for x in df[col].values:
        if not x:
            continue
        for tok in x:
            s.add(tok)
    return sorted(s)

In [3]:
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent   
DATA_DIR = PROJECT_ROOT / "data"

In [4]:
df = pd.read_csv(DATA_DIR/'combined_variants_results_dataset.csv')
df

,id,title,actions,type,type_2
0,02266aadbd,""" Cheeeezy"" Potatoes","['mix', 'add', 'mix', 'add', 'add', 'mix', 'po...",principal,variante_principale
1,02266aadbd,""" Cheeeezy"" Potatoes","['shred', 'mix', 'add', 'mix', 'add', 'add', '...",secondaire,variante_ingredients
2,02266aadbd,""" Cheeeezy"" Potatoes","['mix', 'add', 'add', 'mix', 'add', 'mix', 'po...",secondaire,variante_permutation
3,02266aadbd,""" Cheeeezy"" Potatoes","['mix', 'add', 'mix', 'add', 'add', 'mix', 'to...",secondaire,variante_permutation
4,00ffdd1746,""" It Smells Like Christmas"" Pumpkin Bread","['grease', 'mix', 'blend', 'mix', 'blend', 'fo...",principal,variante_principale
...,...,...,...,...,...
4073482,ffc69d9b0f,tofu/cortege cheese saute,"['cut', 'boil', 'put', 'boil', 'heat', 'put', ...",secondaire,variante_permutation
4073483,ff5b061548,tomato sandwiches w/ lemon pepper mayo,"['prepare', 'add', 'mix', 'slice', 'spread', '...",principal,variante_principale
4073484,ff5b061548,tomato sandwiches w/ lemon pepper mayo,"['slice', 'prepare', 'add', 'mix', 'slice', 's...",secondaire,variante_ingredients
4073485,ff5b061548,tomato sandwiches w/ lemon pepper mayo,"['prepare', 'add', 'mix', 'slice', 'spread', '...",secondaire,variante_permutation


In [5]:
df = data_cleaning_before_test(df,93)


→ Nettoyage de la colonne actions...

Nettoyage de la colonne 'actions'...
✅ Nettoyage terminé!
  ✓ Nettoyage terminé: 4,073,487 lignes
→ Netoyage et Normalisation des verbes...
🚀 Début du nettoyage optimisé...
   Dataset: 4,073,487 lignes
   Actions à supprimer: 2649
   Règles de normalisation: 2300
   Seuil de similarité: 93

📋 Étape 1/3: Extraction des actions uniques...
   ✓ 5,186 actions uniques trouvées

🧹 Étape 2/3: Nettoyage des actions uniques...
   ✓ 2,537 actions conservées
   ✓ 2,649 actions supprimées (51.1%)

⚡ Étape 3/3: Application du mapping...

✅ Terminé en 58.14 secondes!
   Vitesse: 70,059 lignes/seconde

📊 Statistiques:
   Actions moyennes AVANT: 11.95
   Actions moyennes APRÈS: 10.54
   Réduction: 11.8%
→ Suppression des doublons par ID...
  ✓ final_df: 1,510,020 doublons supprimés (2,563,467 restantes)
  ✓ Netoyage et Normalisation terminés


In [11]:
data = pd.read_csv(DATA_DIR/'combined_variants_results_dataset.csv')
print("→ Nettoyage de la colonne actions...")
data  = convert_actions_column_elements(data )
print(f"  ✓ Nettoyage terminé: {len(data ):,} lignes")
        

→ Nettoyage de la colonne actions...

Nettoyage de la colonne 'actions'...
✅ Nettoyage terminé!
  ✓ Nettoyage terminé: 4,073,487 lignes


In [18]:
verbes_avant = extraire_verbes_uniques_from_col(data, "actions")
print("nombre verbes avant nettoyage:", len(verbes_avant))
with open('echantillon_verbes_uniques.txt', 'w', encoding='utf-8') as f:
    for verbe in verbes_avant:
        f.write(verbe + '\n')
print(verbes_avant[:100])

nombre verbes avant nettoyage: 5186
['', ' cook', ' sear', ' sprinkle', ' stir', ' whisk', 'Add', 'BBQ', 'Bring', 'CORRECTED MAIN SEQUENCE', 'Celsius', 'Coca', 'Cool', 'Dig', 'ENJOY', 'Fill', 'French cut', 'Frenched', 'Garnish', 'Heat', 'Ladle', 'Make', 'Microwave', 'OR', 'PAM', 'PULSE', 'Pam', 'Parmesan cheese', 'Place', 'Preheat', 'Roast', 'Run', 'S&F', 'SERVE', 'Season', 'Splenda', 'Spread', 'Stir', 'Tabasco', 'Turn', 'X', 'a', 'aan', 'abschmecken', 'abschäumen', 'absorb', 'absorbed', 'accent', 'accommodate', 'accompanied', 'accompany', 'accordion', 'ace', 'achieve', 'acidulate', 'acquaint', 'acquire', 'act', 'activate', 'ad', 'adapt', 'add', 'add TVP', 'add alternately', 'add back', 'add basil', 'add beans', 'add bell pepper', 'add cheese', 'add collards', 'add crust', 'add dry', 'add eggs', 'add flavoring', 'add flour', 'add frozen vegetables', 'add garlic', 'add glaze', 'add gradually', 'add honey', 'add hot sauce', 'add in', 'add jalapeno', 'add milk', 'add mushrooms', 'add nect

In [7]:

verbes_apres = extraire_verbes_uniques_from_col(df, "actions")
print("nombre verbes avant nettoyage:", len(verbes_apres))
with open('echantillon_verbes_uniques.txt', 'w', encoding='utf-8') as f:
    for verbe in verbes_apres:
        f.write(verbe + '\n')
print(verbes_apres[:100])

nombre verbes avant nettoyage: 642
['', 'absorb', 'acidulate', 'activate', 'add', 'adhere', 'adjust', 'aerate', 'age', 'agitate', 'air dry', 'air fry', 'alternate', 'amalgamate', 'anoint', 'apply', 'arrange', 'assemble', 'attach', 'autolyse', 'bag', 'baggie', 'bake', 'barbecue', 'baste', 'bathe', 'batten', 'batter', 'beard', 'beat', 'behead', 'bind', 'blacken', 'blanch', 'blanket', 'blast', 'blaze', 'blend', 'blenderize', 'blister', 'blitz', 'blob', 'blot', 'blow', 'boil', 'bone', 'braid', 'braise', 'break', 'brew', 'brine', 'broil', 'brown', 'bruise', 'brunoise', 'brush', 'brûlée', 'bubble', 'buff', 'build', 'bundle', 'bung', 'burn', 'burnish', 'burst', 'butcher', 'butter', 'butterfly', 'can', 'candy', 'cap', 'caramelize', 'carbonate', 'carve', 'cavity', 'center', 'char', 'chiffonade', 'chill', 'chip', 'chisel', 'chop', 'chuck', 'chunk', 'churn', 'churn freeze', 'clarify', 'claw', 'clean', 'cleanse', 'cleave', 'clip', 'clump', 'cluster', 'coagulate', 'coat', 'coddle', 'coil', 'color',

In [8]:
with open('echantillon_verbes_uniques.txt', 'w', encoding='utf-8') as f:
    for verbe in verbes_apres:
        f.write(verbe + '\n')

In [9]:
import json
import re
from rapidfuzz import fuzz, process


# =========================
# Configuration
# =========================

THRESHOLD = 93

VERBS_FILE = "echantillon_verbes_uniques.txt"
REMOVE_FILE = "actions_to_remove_final.txt"
NORM_DICT_FILE = "normalization_dict_final.json"

OUT_TEST1 = "test1.txt"
OUT_TEST2 = "test2.txt"


# =========================
# Utils
# =========================

def normalize_surface(text: str) -> str:
    """
    Normalisation de surface :
    - lowercase
    - tirets → espaces
    - espaces multiples → 1 espace
    """
    text = str(text).lower().strip()
    text = text.replace("-", " ")
    text = re.sub(r"\s+", " ", text)
    return text


def load_txt(path):
    with open(path, encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]


# =========================
# Chargement des données
# =========================

verbs = load_txt(VERBS_FILE)
actions_to_remove = load_txt(REMOVE_FILE)

with open(NORM_DICT_FILE, encoding="utf-8") as f:
    normalization_dict = json.load(f)


# Versions normalisées (mapping original → normalisé)
verbs_norm = {v: normalize_surface(v) for v in verbs}
remove_norm = {r: normalize_surface(r) for r in actions_to_remove}
norm_keys_norm = {k: normalize_surface(k) for k in normalization_dict.keys()}


# =========================
# Fuzzy matching
# =========================

def find_fuzzy_matches(source_map, target_map, threshold=95):
    """
    source_map  : dict original → normalisé
    target_map  : dict original → normalisé

    retourne :
    [(source_original, target_original, score), ...]
    """
    results = []

    target_norm_values = list(target_map.values())
    target_original_keys = list(target_map.keys())

    for src_orig, src_norm in source_map.items():
        match, score, idx = process.extractOne(
            src_norm,
            target_norm_values,
            scorer=fuzz.token_sort_ratio
        )

        if score >= threshold:
            tgt_orig = target_original_keys[idx]
            results.append((src_orig, tgt_orig, score))

    return results


# =========================
# Comparaisons
# =========================

matches_test1 = find_fuzzy_matches(verbs_norm, remove_norm, THRESHOLD)
matches_test2 = find_fuzzy_matches(verbs_norm, norm_keys_norm, THRESHOLD)


# =========================
# Écriture des résultats
# =========================

with open(OUT_TEST1, "w", encoding="utf-8") as f:
    for v, r, s in matches_test1:
        f.write(f"{v}  <=>  {r}  (score={s})\n")

with open(OUT_TEST2, "w", encoding="utf-8") as f:
    for v, k, s in matches_test2:
        f.write(f"{v}  <=>  {k}  (score={s})\n")


print(f"✔ {len(matches_test1)} correspondances écrites dans {OUT_TEST1}")
print(f"✔ {len(matches_test2)} correspondances écrites dans {OUT_TEST2}")


✔ 0 correspondances écrites dans test1.txt
✔ 85 correspondances écrites dans test2.txt


In [25]:
def afficher_matches_imperfaits(fichier, seuil_max=100):
    """
    Lit un fichier de matches et affiche uniquement les lignes
    où le score est strictement inférieur au seuil donné.
    
    Args:
        fichier: chemin vers le fichier de matches
        seuil_max: score maximum (défaut: 100)
    """
    matches_imperfaits = []
    
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            # Extraction du score avec regex
            match = re.search(r'score=(\d+)', ligne)
            if match:
                score = int(match.group(1))
                if score < seuil_max:
                    matches_imperfaits.append((ligne.strip(), score))
    
    # Affichage
    print(f"\n{'='*70}")
    print(f"Matches avec score < {seuil_max} : {len(matches_imperfaits)} trouvés")
    print(f"{'='*70}\n")
    
    #for ligne, score in matches_imperfaits:
    #    print(ligne)
    
    return matches_imperfaits


# Utilisation
afficher_matches_imperfaits(OUT_TEST2)

# Ou avec un seuil différent
# afficher_matches_imperfaits(OUT_TEST2, seuil_max=98)



Matches avec score < 100 : 49 trouvés



[('Garnish  <=>  guarnish  (score=93.33333333333333)', 93),
 ('Microwave  <=>  micro wave  (score=94.73684210526316)', 94),
 ('Preheat  <=>  presheat  (score=93.33333333333333)', 93),
 ('baseline  <=>  base-line  (score=94.11764705882352)', 94),
 ('brunoise  <=>  bruinoise  (score=94.11764705882352)', 94),
 ('buttercream  <=>  cream butter  (score=95.65217391304348)', 95),
 ('caramelize  <=>  carmelize  (score=94.73684210526316)', 94),
 ('chiffonade  <=>  chiffionade  (score=95.23809523809523)', 95),
 ('combine  <=>  combined  (score=93.33333333333333)', 93),
 ('crosscut  <=>  cross cut  (score=94.11764705882352)', 94),
 ('crosshatch  <=>  cross-hatch  (score=95.23809523809523)', 95),
 ('crumble  <=>  crumbled  (score=93.33333333333333)', 93),
 ('crumple  <=>  crumpled  (score=93.33333333333333)', 93),
 ('defrost  <=>  de-frost  (score=93.33333333333333)', 93),
 ('deglaze  <=>  de-glaze  (score=93.33333333333333)', 93),
 ('degrease  <=>  de-grease  (score=94.11764705882352)', 94),
 ('d